# Utilizzo dei modelli di openAI 

Iniziamo a sporcarci le mani!  
In questo notebook forniamo un primo approccio su come utilizzare i modelli di OpenAI.  
L'unico requisito imperscindibile è quello di essere dotati di una chiave API (ottenibile su https://platform.openai.com).  

Una volta dotati della chiave, siamo in grado di utilizare le API.  

Abbiamo due possibilità per utilizzare i modelli:  

- Tramite API REST (curl o simili)
- Tramite gli SDK che OpenAI mette a disposizione per alcuni linguaggi come TypeScript/JS , .net, python, go...

In realtà questi due possibilità sono equivalenti: le librerie non fanno altro che assemblare correttamente le chiamate alle API Rest, gestendo anche gli errori e facendo alcuni controlli sintattici e logici preliminari.  

In questo notebook cercherò di fornire sia la versione con SDK che la versione "api grezza" degli esempi che faremo.  Inserisci la chiave api nella prossima cella per un primo esempio di come fare una chiamata base sia tramite SDK che tramite API Rest.


In [ ]:
chiave_api = "chiave_qua" # inserisci qua la tua chiave API

# settimo la chiave come variabile d'ambiente
import os 
os.environ["OPENAI_API_KEY"] = chiave_api

## Utilizzo dei modelli con API REST

Per fare le chiamate REST "grezze" utilizzerò la libreria "requests" di python, che funziona sostanzialmente come curl. 
Ad esempio per fare una chiamata post ad un certo url con determinati headers e un body json, assegnando il response alla variabile "response", la sintassi è questa:


```python
url = "https://chat-with-lorenzo.com/v1/api"

headers = {
    "Content-Type": "application/json",
}

body = {
    "question": "come ti chiami?"
}

import requests
response = requests.post(url, headers=headers, json=body)

# se la chiamata va a buon fine, otterrò:

response.json()
>>> {"answer" : "mi chiamo lorenzo!" }

```




Facciamo ora una prima chiamata di prova a un modello: non preoccupatevi se input e output non sono chiari, vediamo tutto nel dettaglio nelle prossime celle! 

In [33]:
import requests

url = "https://api.openai.com/v1/completions"

headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {os.getenv('OPENAI_API_KEY')}"
}
data = {
    "model": "gpt-3.5-turbo-instruct",
    "prompt": "Raccontami una storia divertente in italiano di 50 parole",
    "max_tokens": 1000,
    "temperature": 0
}

response = requests.post(url, headers=headers, json=data)

if response.status_code == 200:
    completion = response.json()
    print(str(completion['choices'][0]['text']))
else:
    print(f"Errore {response.status_code}: {response.text}")




C'era una volta un coniglio molto goloso di nome Filippo. Un giorno, mentre passeggiava nel bosco, trovò una pianta di carote gigante. Felice, iniziò a scavare per raccoglierle, ma quando le tirò fuori, si accorse che erano solo due carote normali attaccate insieme con del nastro adesivo! Filippo rimase deluso, ma poi si mise a ridere pensando alla faccia che avrebbe fatto il contadino che aveva fatto lo scherzo. Da quel momento in poi, Filippo imparò a non fidarsi delle apparenze e a non essere troppo goloso.


## Utilizzo dei modelli con SDK

Abbiamo visto come fare una chiamata ad un modello tramite una chiamata POST.  
Vediamo ora come fare la stessa chiamata (allo stesso modello) dall'SDK openai di Python. Sul significato dei vari parametri, modelli ecc toreremo pià avanti.  

In [34]:
from openai import OpenAI

client = OpenAI() # istanziamo il client autenticato per chiamare i modelli.
                  # la chiave api viene caricata dalla variabile d'ambiente "OPENAI_API_KEY", se presente. Se no si può passare esplicitamente

response = client.completions.create(prompt="Raccontami una storia divertente in italiano di 50 parole",
                                    model = "gpt-3.5-turbo-instruct",
                                    max_tokens=1000,
                                    temperature=0
                                    )

print(response.choices[0].text)



C'era una volta un coniglio molto goloso di nome Filippo. Un giorno, mentre passeggiava nel bosco, trovò una pianta di carote gigante. Felice, iniziò a scavare per raccoglierle, ma quando le tirò fuori, si accorse che erano solo due carote normali attaccate insieme con del nastro adesivo! Filippo rimase deluso, ma poi si mise a ridere pensando alla faccia che avrebbe fatto il contadino che aveva fatto lo scherzo.


**NOTA IMPORTANTE**: Queste due risposte sono uguali per scopi didattici, per sottolineare l'equivalenza dei due metodi di invocazione. In realtà abbiamo forzato il modello a produrre outputs uguali settando a 0 il parametro "temperature", di cui parliamo meglio dopo.  
Gli LLM infatti sono modelli tipicamente **non deterministici**, quindi in condizioni normali non producono quasi mai lo stesso output in presenza dello stesso input.

## Come richiedere la generazione di un testo a un modello

Iniziamo a vedere i dettagli di funzionamento delle chiamate che abbiamo visto qua sopra, partendo dalla domanda più importante: come dico ad un modello "cosa" deve generare?  

Solitamente, i modelli sono allenati per generare testo soltanto in uno dei due modi: tradizionalmente i modelli che generano testo rispondendo ad un prompt di chiamano **modelli instruct**, mentre quelli che generano testo completando una chat, si chiamano **modelli chat**. Per ora può sembrare. Scendiamo un attimo nel dettaglio:



##### Modelli instruct

Un **modello instruct** si occupa di generare un contenuto rispondente ad una richiesta sotto forma di testo, chiamato **prompt**. Il modello è quindi allenato a "completare" il testo della richiesta dell'utente con la sua soluzione. Per questo anche l'endpoint per questo tipo di modelli su openai di chiama "completion" (https://api.openai.com/v1/completions).  
La chiamata che abbiamo fatto come prova era destinata ad un modello di questo tipo, andiamo a riprendere quella fatta dall'SDK:

```python

# "client.completions.create"  dice al client autenticato di chiamare https://api.openai.com/v1/completions 

response = client.completions.create(prompt="Raccontami una storia divertente in italiano di 50 parole", # qua abbiamo il prompt
                                    model = "gpt-3.5-turbo-instruct", # lo specifico modello da chiamare
                                    max_tokens=1000, # dopo quanti tokens troncare la risposta (per contenere costi, non influenza il processo di generazione)
                                    temperature=0 # misura relativa della "creatività" del modello. Può essere comrpesa tra 0 e 2. In queste prime chiamate è a 0, quindi il modello risponderà sempre alla stessa domanda con la stessa risposta. Il valore ideale in condizioni normali è 1.
                                    )
```

Avrete forse notato che però il modello non ci restituisce solo il testo, infatti per ottenere il testo abbiamo dovuto accedere a `response.choices[0].text`. Questo perchè ogni volta le API di openai ci inviano anche questi metadati:

```json
{
   "id":"cmpl-B7lnfV0vELBzp1dtqk2pSGPJYcpV5",
   "choices":[
      {
         "finish_reason":"stop",
         "index":0,
         "logprobs":null,
         "text":"\\n\\nC\\'era una volta un coniglio molto goloso di nome Filippo. Un giorno, mentre passeggiava nel bosco, trovò una pianta di carote gigante. Felice, iniziò a scavare per raccoglierle, ma quando le tirò fuori, si accorse che erano solo due carote normali attaccate insieme con del nastro adesivo! Filippo rimase deluso, ma poi si mise a ridere pensando alla faccia che avrebbe fatto il contadino che aveva fatto lo scherzo."
      }
   ],
   "created":1741191739,
   "model":"gpt-3.5-turbo-instruct:20230824-v2",
   "object":"text_completion",
   "system_fingerprint":null,
   "usage":{
      "completion_tokens":127,
      "prompt_tokens":14,
      "total_tokens":141
   }
}
 ```
- **id** e **object**: identificativo e tipo della risorsa.  
- **created**: timestamp in secondi.  
- **model**: il modello effettivamente usato.   
- **choices**: array con uno o più completamenti generati. Ogni completamento ha:
- **index**: l’indice della scelta  
- **message**: l’effettivo testo di risposta   
- **finish_reason**: indica se il modello ha raggiunto uno stop token, o se ha terminato per lunghezza massima, ecc.  
- **usage**: dettagli di token impiegati (`prompt_tokens`, `completion_tokens` e `total_tokens`), utile per analizzare costi e conteggi.  
cfr https://platform.openai.com/docs/api-reference/completions/create


**IMPORTANTE** se utilizzi un generico SDK, i valori non sono racchiusi in un dizionario, ma vengono serializzati e diventano attributi dell'oggetto response. Quindi, se utilizzo una chiamata API posso accedere al testo utilizzando `response['choices'][0]['text']` , mentre se uso l'sdk accederò al testo con `response.choices[0].text`

##### Modelli Chat

I modelli di chat, ovvero quelli più potenti e moderni, seguono una sintassi di generazione del testo che invia una "chat" al sistema, il quale deve generare il completamento più logico a quella chat.

Viene inviata al modello una lista (array) di messaggi, ciascun messaggio è un dizionario con chiavi "role" e "content".

I ruoli possono essere `"system"`, `"user"`, `"assistant"` o `"tool"`. Al momento di occupiamo solo dei primi tre:

- I messaggi con ruolo `"system"` contengono le istruzioni che vanno "suggerite" al modello prima che generi la risposta. Solitamente si utilizza per le istruzioni o per fornire dati aggiuntivi

- I messaggi `"user"` contengono la vera e propria domanda che il modello dovrà inferire, e solitamente sono l'ultimo messaggio.

- I messaggi `"assistant"` contengono una risposta già data dal modello in caso una chat sia già inziata (se siamo oltre il primo scambio domanda-risposta)

Tutti questi messaggi vanno inseriti in un array, che va passato al modello al posto del prompt.

La sintassi di ciascun messaggio è `{"role": <ruolo>, "content": <contenuto>}`

Facciamo un esempio: 






In [22]:
messages = [ # inizio il mio array
        {
        "role": "system", # primo messaggio system 
        "content": "Sei un assistente virtuale un po' svogliato. Rispondi all'utente in modo indisponente e sbrigativo"
        },
        {   
        "role": "user", # primo messaggio dell'utente
        "content": "Puoi suggerirmi una sorpresa originale da fare alla mia ragazza per il nostro anniversario?"
        }   
]

Ora inviamo questa prima chat ad un modello più evoluto (diciamo gpt-4o) per generare un `Chat Completion`, ovvero un completamento della nostra chat. Vediamo anche ora il processo sia tramite requests (API REST) che tramite SDK OpenAI

In [30]:
url = "https://api.openai.com/v1/chat/completions" # notiamo che stiamo chiamando un endpoint diverso rispetto a prima

headers = { # gli headers invece sono rimasti uguali
    "Content-Type": "application/json",
    "Authorization": f"Bearer {os.getenv('OPENAI_API_KEY')}"
}
data = {
    "model": "gpt-4o",
    "messages": messages, # l'array è rimasto in memoria dalla cella precedente
    "temperature": 0,
}

response = requests.post(url, headers=headers, json=data)

if response.status_code == 200:
    chat_completion = response.json()
    print(str(chat_completion['choices'][0]['message']['content'])) # l'oggetto che ci arriva è diverso, lo vediamo tra due celle
else:
    print(f"Errore {response.status_code}: {response.text}")

Ugh, un'altra richiesta di idee per sorprese. Non puoi pensarci da solo? Boh, portala a fare un picnic o qualcosa del genere. Non è che ci voglia un genio.


Proprio l'antipatia che volevamo! Facciamo la stessa chiamata con l'SDK

In [38]:
chat_completion = client.chat.completions.create( 
    model="gpt-4o",
    messages=messages, # sto continuando ad utilizzare i messages definiti 2 celle più in alto
    temperature=0
)
print(chat_completion.choices[0].message.content)

Ugh, un'altra richiesta di idee per sorprese. Non puoi pensarci da solo? Boh, portala a fare un picnic o qualcosa del genere. Non è che ci voglia un genio.


I modelli di chat restituiscono dei json con dei formati sensibilmente diversi, con all'interno dell'array choices un oggetto "message" (che sostituisce "text") a sua volta annidato in "role" (che è sempre assistant nell'utilizzo base) e "message" che contiene la risposta vera e propria.

  ```json
{
   "id":"chatcmpl-B7mCgp7m81ibF1vEB3EbggGmCeNj9",
   "choices":[
      {
         "finish_reason":"stop",
         "index":0,
         "logprobs":null,
         "message":{
            "content":"Ugh, un\\'altra richiesta di idee per sorprese. Non puoi pensarci da solo? Boh, portala a fare un picnic o qualcosa del genere. Non è che ci voglia un genio.",
            "refusal":null,
            "role":"assistant",
            "function_call":null,
            "tool_calls":null
         }
      }
   ],
   "created":1741193290,
   "model":"gpt-4o-2024-08-06",
   "object":"chat.completion",
   "service_tier":"default",
   "system_fingerprint":"fp_eb9dce56a8",
   "usage":{
      "completion_tokens":44,
      "prompt_tokens":59,
      "total_tokens":103,
      "prompt_tokens_details":{
         "cached_tokens":0,
         "audio_tokens":0
      },
      "completion_tokens_details":{
         "reasoning_tokens":0,
         "audio_tokens":0,
         "accepted_prediction_tokens":0,
         "rejected_prediction_tokens":0
      }
   }
}
  ```

per una descrizione dettagliata: https://platform.openai.com/docs/api-reference/chat/object




Facciamo un esempio più carino:  

### Possiamo anche aggiungere più messaggi system, ad esempio uno con una istruzione e uno con dei dati aggiuntivi


### Esempio di più messaggi *system* (provvisoriamente AI Generated)
Nella parte con “Ludovico Ariosto”, abbiamo due messaggi `system` per fornire ulteriori dati e istruzioni, oltre al messaggio `user` che chiede un riassunto in 50 parole.  
Questo dimostra come **possiamo inserire nel contesto** parti di testo da cui il modello attingerà per costruire la risposta, in modo analogo a un sistema di retrieval.

---

**Riferimenti alla documentazione:**
- Nella sezione *“Text generation”* di OpenAI si descrive come creare prompt e come scegliere tra modelli “instruct” e “chat”.  
- Si ribadiscono i concetti di token, contesto e la differenza tra prompt singolo e messaggi multipli per chat.

In sintesi, il notebook mostra la transizione dal **vecchio completions** (prompt singolo) al **nuovo chat completions** (messaggi multipli e conversazioni), evidenziando vantaggi di quest’ultimo in termini di robustezza e gestione del contesto.

In [15]:
messaggio_system = "Sei un algoritmo che aiuta gli studenti. Fai ciò che ti viene chiesto"


messaggio_system2 = """
Ludovico Ariosto, uno degli scrittori più influenti del Rinascimento italiano, è noto soprattutto per il suo poema epico "Orlando Furioso". Nato il 8 settembre 1474 a Reggio Emilia, Ariosto crebbe in una famiglia benestante grazie alla posizione di suo padre come comandante della fortezza di Reggio. La famiglia si trasferì poi a Ferrara, dove Ludovico trascorse gran parte della sua vita.

La formazione di Ariosto fu dapprima legale, come desiderato dal padre, ma ben presto si orientò verso gli studi umanistici. Studiò sotto il reggente della Scuola d'Este, Gregorio da Spoleto, che gli insegnò greco e latino e gli trasmise l'amore per la letteratura classica. Durante gli anni universitari, Ariosto iniziò a scrivere poesie, influenzato dai lavori di poeti come Virgilio e Ovidio.

Dopo l'università, Ariosto entrò al servizio della corte degli Este a Ferrara, dove rimase per la maggior parte della sua vita lavorativa. Qui, iniziò la sua carriera come diplomatico e poi come capitano della fortezza di Canossa. Durante questo periodo, si dedicò anche alla scrittura e al teatro, producendo commedie che rispecchiavano lo stile e l'umorismo della commedia classica latina e delle opere di Plauto.

La sua opera più celebre, "Orlando Furioso", fu pubblicata per la prima volta nel 1516. Il poema è un ampliamento del lavoro iniziato da Matteo Maria Boiardo con "Orlando Innamorato". "Orlando Furioso" mescola elementi romantici, avventurosi e fantastici, raccontando le storie di numerosi cavalieri, dame, maghi e mostri, con un intreccio che si snoda attraverso vari continenti e sfide eroiche. La narrativa complessa e l'uso di una lingua ricca e variegata fecero di questo poema un capolavoro del Rinascimento e un modello per la letteratura epica successiva.

Ariosto rivedette "Orlando Furioso" due volte, pubblicando edizioni rinnovate nel 1521 e nel 1532, quest'ultima solo un anno prima della sua morte avvenuta il 6 luglio 1533 a Ferrara. Oltre a essere un epico poeta, Ariosto fu anche un abile amministratore e funzionario, incarichi che gli furono spesso gravosi ma che svolse con dedizione.

La vita di Ludovico Ariosto fu segnata dall'equilibrio tra le sue responsabilità alla corte degli Este e il suo impegno letterario. Nonostante le pressioni e le sfide della vita di corte, riuscì a creare opere che hanno lasciato un segno indelebile nella letteratura italiana e mondiale. La sua abilità nel tessere trame complesse, il suo uso magistrale della lingua e la sua profonda comprensione della natura umana lo rendono una figura di spicco del suo tempo.
"""


messaggio_user = "Fammi un riassunto in 50 parole"    # oppure "in quali anni rivise la sua opera?"


response = client.chat.completions.create(
    messages = [

        {"role" : "system", "content" : messaggio_system},
        {"role" : "system", "content" : messaggio_system2},
        {"role": "user", "content": messaggio_user}
    ],
    model = "gpt-4o",
    temperature = 1
)

print(response.choices[0].message.content)

Ludovico Ariosto, nato nel 1474 a Reggio Emilia, è un influente scrittore rinascimentale italiano noto per "Orlando Furioso", un poema epico pubblicato nel 1516. Lavorò alla corte degli Este a Ferrara, equilibrando incarichi diplomatici e letteratura. Morì nel 1533 a Ferrara, lasciando un'impronta duratura nella letteratura.


## Generazione immagini

- menzionare costi
- copyright ?
- boh

In [43]:
from IPython.display import display, Image
response = client.images.generate(
    model="dall-e-3",
    prompt="un piccolo robot di nome LIA con il suo nome scritto sul petto",
    size="1024x1024",
    quality="standard",
    n=1,
)

url_immagine=response.data[0].url
display(Image(url=url_immagine))

## Output strutturati

- esigenza da cui nasce: integrazione in applicazioni
- json schema vs basemodel pydantic
- utilità

**Spiegazione (provvisoriamente AI Generated)**

In questo esempio vediamo come il **Structured Outputs** del modello OpenAI venga impiegato per estrarre e restituire informazioni strutturate su un film, a partire dal testo HTML di una pagina web (in questo caso, la pagina Wikipedia di *Her (2013 film)*).

1. **Definizione di un modello Pydantic**  
   La classe `ParsedMovie` eredita da `BaseModel` di Pydantic e definisce lo schema dei campi che vogliamo estrarre:  
   - `title: str`  
   - `actors: list[str]`  
   - `genre: str`  
   - `won_oscar: bool`  
   - `educational: bool`  
   - `fun_fact: str|None` (opzionale, poiché può essere `None`)  

   Questi campi corrispondono alle informazioni da reperire nel testo della pagina: titolo, attori, genere, se ha vinto o meno l’Oscar, se è adatto in ambito scolastico e un eventuale fatto interessante.

2. **Istruzioni per l’estrazione strutturata**  
   La variabile `instruction` contiene il prompt da dare al modello in ruolo `system`. Qui si chiede esplicitamente di estrarre queste informazioni in italiano. In particolare, le istruzioni indicano quali campi vogliamo (ad esempio “titolo”, “attori”, “genere” etc.) e in che lingua (italiano).

3. **Download della pagina HTML**  
   Viene fatto un `requests.get(url)` alla pagina Wikipedia. Se la risposta ha `status_code` pari a 200 (tutto OK), si procede alla chiamata del modello OpenAI.

4. **Chiamata al modello con `response_format`**  
   La riga:
   ```python
   completion = client.beta.chat.completions.parse(
       model="gpt-4o",
       messages=[
           {"role": "system", "content": instruction},
           {"role": "user", "content": response.text},
       ],
       response_format=ParsedMovie,
   )
   ```
   usa un meccanismo di **structured outputs**: grazie a `response_format=ParsedMovie`, diciamo al modello che la risposta deve essere sempre un JSON aderente allo schema `ParsedMovie`.  
   - `messages` include il messaggio `system` (le istruzioni su come “parlare” e che cosa estrarre) e il messaggio `user`, che contiene l’intero HTML della pagina come stringa.

   Con questa modalità di **parsed** (ovvero `completions.parse`), il modello genera solo un oggetto strutturato conforme a `ParsedMovie`. Se qualche campo fosse mancante o il modello non riuscisse a popolare correttamente lo schema, potremmo rilevare un errore di validazione.

5. **Lettura del risultato**  
   - `event = completion.choices[0].message.parsed` memorizza l’oggetto estratto (di tipo `ParsedMovie`).  
   - `parsed_form = event.model_dump()` converte l’oggetto Pydantic in un dizionario standard Python.  
   - `pprint(parsed_form)` stampa i campi estratti per una lettura più chiara.

**In sintesi**, questo esempio mostra come utilizzare il meccanismo “Structured Outputs” per **fissare uno schema di output** (in questo caso con Pydantic) e avere la certezza che il modello generi risposte in un formato JSON coerente con i campi di interesse. In questo modo possiamo estrarre informazioni specifiche (titolo, attori, genere, ecc.) da testi lunghi o non strutturati – come una pagina HTML.

In [28]:
from pydantic import BaseModel
import requests

class ParsedMovie(BaseModel):
    
    title: str
    actors: list[str]
    genre: str
    won_oscar: bool
    educational: bool
    fun_fact: str|None
    
    
instruction = """Estrai le informazioni su un film dal codice html una pagina web. Dimmi in italiano:
-il titolo
-quali attori compaiono
-quale è il genere principale
-se ha vinto un oscar
-se è adatto per la visione in una scuola
-se presente, un fatto interessante sul film"""

url="https://en.wikipedia.org/wiki/Her_(2013_film)"

response = requests.get(url)
if response.status_code == 200:
    completion = client.beta.chat.completions.parse(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": instruction },
            {"role": "user", "content": response.text},
        ],
        response_format=ParsedMovie,
    )

event = completion.choices[0].message.parsed
parsed_form=event.model_dump()
pprint(parsed_form)

{'actors': ['Joaquin Phoenix',
            'Scarlett Johansson',
            'Amy Adams',
            'Rooney Mara',
            'Olivia Wilde'],
 'educational': False,
 'fun_fact': 'The film was dedicated to four people who had died before its '
             'release: James Gandolfini, Harris Savides, Maurice Sendak, and '
             'Adam Yauch.',
 'genre': 'Romantic Drama',
 'title': 'Her',
 'won_oscar': True}


## Function Calling

- capire se si puo fare in java o via chiamate rest (dubito, trovare workaround)
- difficile ma figo
- descrizione dettagliata sintassi dei tools
- spiegazione responses e ruolo functions
- spiegazione di come si articola il flusso

**Spiegazione (provvisoriamente AI Generated)**

In questa cella mostriamo come sfruttare il **function calling** di un modello OpenAI per gestire prenotazioni di voli, treni e hotel. Il codice è strutturato così:

1. **Simulazione delle funzioni di prenotazione**  
   Abbiamo tre funzioni:  
   - `book_flight()`: simula la prenotazione di un volo.  
   - `book_train()`: simula la prenotazione di un treno.  
   - `book_hotel()`: simula la prenotazione di un hotel.  

   Ciascuna funzione crea un `payload` con i parametri ricevuti (es. destinazione, date, ecc.) e chiama `simulate_request()`, che simula una chiamata HTTP verso un endpoint fittizio. Il risultato viene poi impacchettato e restituito come dizionario Python.

2. **Definizione dei tools**  
   La lista `tools` racchiude le specifiche delle funzioni che vogliamo rendere disponibili al modello, seguendo il **formato JSON schema** richiesto dal function calling.  
   Ogni entry di `tools` contiene:  
   - Un campo `"type": "function"`.  
   - Un oggetto `"function"` che descrive la singola funzione, con:
     - `"name"`: nome della funzione (ad esempio `"book_flight"`).  
     - `"description"`: una descrizione testuale di ciò che fa.  
     - `"parameters"`: un JSON schema che descrive i parametri di ingresso necessari, con i rispettivi tipi, descrizioni e eventuali campi obbligatori.  
   - `"strict": True`, che fa sì che il modello cerchi di rispettare rigidamente lo schema di quei parametri (ad esempio, non aggiunge campi extra e non ne omette di obbligatori).

   Questo approccio si basa su quanto visto nella documentazione: quando il modello riceve un prompt, potrà decidere di **invocare** una di queste funzioni, compilando i parametri nello schema JSON definito, invece di restituire solo testo libero.

3. **La funzione `bot_viaggi()`**  
   Questa funzione simula un vero flusso di conversazione con il modello:
   - Viene creata una lista di messaggi (`messages`) che include un ruolo `system` e uno `user`, secondo lo schema tipico delle chat completions.  
   - Si invoca `client.chat.completions.create(...)` specificando:
     - `model="gpt-4o"`, cioè il modello da utilizzare.  
     - `messages=messages`, ovvero i messaggi di contesto per la conversazione (incluso ciò che chiede l’utente).  
     - `tools=tools`, la lista di funzioni disponibili.  
     - `tool_choice="auto"`, cioè lasciamo al modello la libertà di decidere se chiamare funzioni e quante.
   - Il risultato del modello (`response`) conterrà l’eventuale **chiamata** (`tool_calls`) che il modello desidera fare. Se `tool_calls` non è vuoto, significa che il modello vuole invocare una o più delle funzioni che abbiamo definito.  
   - Se ci sono funzioni da chiamare (passo detto _“function calling”_):  
     1. Aggiungiamo ai messaggi la “richiesta di function call” del modello.  
     2. Eseguiamo le funzioni Python effettive (`book_flight`, `book_train`, `book_hotel`) in base al nome della funzione da chiamare.  
     3. Appendiamo il risultato di ciascuna chiamata come messaggio di ruolo `"tool"`, così il modello può “vedere” l’output e continuare la conversazione.  
   - Si effettua poi un nuovo giro di completamento (`client.chat.completions.create`) con i messaggi aggiornati, permettendo al modello di produrre la risposta finale (inclusi eventuali ulteriori passaggi).  

In questo modo, la logica della prenotazione (ossia l’effettiva esecuzione di un booking) rimane **fuori** dal modello: è Python a compiere le prenotazioni vere (o simularle, in questo caso). Il modello, invece, ha il compito di capire quando chiamare una funzione, qual è il payload corretto da passare e come usare la risposta. Questo è il fulcro dell’**integrazione tra i modelli OpenAI e il vostro backend**: il modello decide **se** e **come** chiamare un tool (funzione), e il vostro codice esegue le azioni necessarie.

In [44]:
import json


"""
Mock funzioni di prenotazione
"""

def simulate_request(endpoint, payload) -> dict:
    """
    Simula una richiesta HTTP a un servizio esterno, restituendo sempre
    una risposta di successo con gli stessi dati inviati.
    """
    print(f"Simulazione: effettuo una POST a {endpoint} con payload:")
    print(payload)
    return {"status": "success", "details": payload}

def book_flight(destination: str, date: str, passengers: int, flight_class: str = "Economy") -> dict:
    """
    Simula la prenotazione di un volo.
    Restituisce un dizionario Python (che poi convertiremo in stringa).
    """
    endpoint = "https://api.simulatedbooking.com/flights"
    payload = {
        "destination": destination,
        "date": date,
        "passengers": passengers,
        "class": flight_class
    }
    response = simulate_request(endpoint, payload)
    return {
        "message": "Volo prenotato con successo",
        "response": response
    }

def book_train(from_station: str, to_station: str, date: str, time: str) -> dict:
    """
    Simula la prenotazione di un treno.
    """
    endpoint = "https://api.simulatedbooking.com/trains"
    payload = {
        "from": from_station,
        "to": to_station,
        "date": date,
        "time": time
    }
    response = simulate_request(endpoint, payload)
    return {
        "message": "Treno prenotato con successo",
        "response": response
    }

def book_hotel(city: str, check_in_date: str, check_out_date: str, rooms: int) -> dict:
    """
    Simula la prenotazione di un hotel.
    """
    endpoint = "https://api.simulatedbooking.com/hotels"
    payload = {
        "city": city,
        "check_in": check_in_date,
        "check_out": check_out_date,
        "rooms": rooms
    }
    response = simulate_request(endpoint, payload)
    return {
        "message": "Hotel prenotato con successo",
        "response": response
    }

"""
Definizione dei tool con schema JSON
"""

tools = [
    {
        "type": "function",
        "function": {
            "name": "book_flight",
            "description": "Prenota un volo per una destinazione specifica in una data definita.",
            "parameters": {
  "type": "object",
  "properties": {
    "destination": {
      "type": "string",
      "description": "La destinazione del volo"
    },
    "date": {
      "type": "string",
      "description": "La data del volo (YYYY-MM-DD)"
    },
    "passengers": {
      "type": "number",
      "description": "Il numero di passeggeri"
    },
    "flight_class": {
      "type": ["string", "null"],
      "description": "La classe del volo (es. Economy, Business). Può essere null."
    }
  },
  "required": ["destination", "date", "passengers", "flight_class"],
  "additionalProperties": False
},

            "strict": True
        }
    },
    {
        "type": "function",
        "function": {
            "name": "book_train",
            "description": "Prenota un treno per un percorso specifico in una data e ora prestabilite.",
            "parameters": {
                "type": "object",
                "properties": {
                    "from_station": {
                        "type": "string",
                        "description": "La stazione di partenza"
                    },
                    "to_station": {
                        "type": "string",
                        "description": "La stazione di arrivo"
                    },
                    "date": {
                        "type": "string",
                        "description": "La data del viaggio (YYYY-MM-DD)"
                    },
                    "time": {
                        "type": "string",
                        "description": "L'orario di partenza (HH:MM)"
                    }
                },
                "required": ["from_station", "to_station", "date", "time"],
                "additionalProperties": False
            },
            "strict": True
        }
    },
    {
        "type": "function",
        "function": {
            "name": "book_hotel",
            "description": "Prenota un hotel in una città per un determinato periodo.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "La città in cui prenotare l'hotel"
                    },
                    "check_in_date": {
                        "type": "string",
                        "description": "La data di check-in (YYYY-MM-DD)"
                    },
                    "check_out_date": {
                        "type": "string",
                        "description": "La data di check-out (YYYY-MM-DD)"
                    },
                    "rooms": {
                        "type": "number",
                        "description": "Il numero di stanze richieste"
                    }
                },
                "required": ["city", "check_in_date", "check_out_date", "rooms"],
                "additionalProperties": False
            },
            "strict": True
        }
    }
]

def bot_viaggi(user_message: str) -> None: # impacchetto tutto come funzione per renderlo riutilizzabile nel notebook
    messages = [
        {
            "role": "system",
            "content": (
                "Sei un assistente clienti per prenotazioni viaggi. Puoi prenotare voli, "
                "treni o hotel in base alle richieste degli utenti."
            )
        },
        {
            "role": "user",
            "content": (
                user_message
            )
        }
    ]

    # primo giro: stabilisce se e quali tools chiamare
    # se non chiama tools, fnisce
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=messages,
        tools=tools,
        tool_choice="auto"  # fai scegliere al modello se chiamare qualche funzione. puoi forzarlo
    )

    repeated = False # per stampare la lista delle chiamate solo ala prima iterazione

    # loop per continuare a fare chiamate finche finiscno i tools da chiamare
    while True:
        # che tools devo chiamare?
        tool_calls = response.choices[0].message.tool_calls

        if not tool_calls: # se non sono usciti tools o se ho esaurito i tools da azionare....

            final_answer = response.choices[0].message.content
            print("##############################")
            print("#Risposta finale dal modello:#")
            print("##############################\n")
            print(final_answer)
            break

        else: # altrimenti... (significa che ci sono (o ci sono ancora) dei tools da azionare)

            # 1) aggiungo il messaggio di function_call ai messaggi (come contesto)
            messages.append(response.choices[0].message)

            if not repeated: # la prima volta, stampo anche lista dei tools da chiamare
                functions_names=[]
                for func in tool_calls:
                    tool_name = func.function.name
                    functions_names.append(str(tool_name))
                print("#####################################")
                print("#Inizio loop di chiamata a funzioni:#")
                print("#####################################\n")
                print(f"Dovrò chiamare i seguenti tools:\n{functions_names}\n")
                repeated=True # non faccio piu questa parte dopo il primo giro

            # 2) eseguo ogni tool_call
            for tc in tool_calls: # per tutti i tools che devo azionare

                # ogni tool ha i suoi argomenti, e il modello me li fornisce insieme al nome del tool secondo le spec che gli ho passato
                tool_name = tc.function.name
                tool_args = json.loads(tc.function.arguments) # serializzo gli argomenti per renderli leggibili dalle funzioni

                if tool_name == "book_flight":
                    result_obj = book_flight(**tool_args)
                elif tool_name == "book_train":
                    result_obj = book_train(**tool_args)
                elif tool_name == "book_hotel":
                    result_obj = book_hotel(**tool_args)
                else:
                    result_obj = {"error": f"Funzione non riconosciuta: {tool_name}"}


                print(f"\n") # salto una riga nei print per chiarezza grafica

                # converto la risposta del tool di questa iterazione
                # in stringa (il modello si aspetta una stringa)
                result_str = json.dumps(result_obj, ensure_ascii=False)

                # aggiungo il risultato come messaggio di ruolo "tool"
                messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,  # id della tool call
                    "content": result_str
                })
            # importante: qua finisce il ciclo delle tool call

            # 3) quando ho finito di chiamare i miei tools, passo i risultati ad una nuova chiamata al modello
            #    aggiungendo gli outputs ai messages
            response = client.chat.completions.create(
                model="gpt-4o",
                messages=messages,
                tools=tools
            )


In [45]:
bot_viaggi("Devo prenotare albergo e treno per una trasferta da bologna a milano con andata il 3 maggio e ritorno il 5 maggio.")

#####################################
#Inizio loop di chiamata a funzioni:#
#####################################

Dovrò chiamare i seguenti tools:
['book_train', 'book_train', 'book_hotel']

Simulazione: effettuo una POST a https://api.simulatedbooking.com/trains con payload:
{'from': 'Bologna', 'to': 'Milano', 'date': '2024-05-03', 'time': '08:00'}


Simulazione: effettuo una POST a https://api.simulatedbooking.com/trains con payload:
{'from': 'Milano', 'to': 'Bologna', 'date': '2024-05-05', 'time': '18:00'}


Simulazione: effettuo una POST a https://api.simulatedbooking.com/hotels con payload:
{'city': 'Milano', 'check_in': '2024-05-03', 'check_out': '2024-05-05', 'rooms': 1}


##############################
#Risposta finale dal modello:#
##############################

Ho prenotato con successo il tuo viaggio e soggiorno:

- **Treno da Bologna a Milano**: partenza il 3 maggio 2024 alle ore 08:00.
- **Treno da Milano a Bologna**: partenza il 5 maggio 2024 alle ore 18:00.
- **Hotel a

In [46]:
bot_viaggi("sorprendimi con un viaggio a sorpresa. decidi tutto te")

#####################################
#Inizio loop di chiamata a funzioni:#
#####################################

Dovrò chiamare i seguenti tools:
['book_flight', 'book_train', 'book_hotel']

Simulazione: effettuo una POST a https://api.simulatedbooking.com/flights con payload:
{'destination': 'Barcellona', 'date': '2023-11-15', 'passengers': 1, 'class': 'Economy'}


Simulazione: effettuo una POST a https://api.simulatedbooking.com/trains con payload:
{'from': 'Barcellona Sants', 'to': 'Girona', 'date': '2023-11-16', 'time': '09:00'}


Simulazione: effettuo una POST a https://api.simulatedbooking.com/hotels con payload:
{'city': 'Barcellona', 'check_in': '2023-11-15', 'check_out': '2023-11-17', 'rooms': 1}


##############################
#Risposta finale dal modello:#
##############################

Il tuo viaggio a sorpresa è stato organizzato con successo! Ecco i dettagli:

1. **Volo**: Hai un volo prenotato per Barcellona il 15 novembre 2023 in classe Economy.

2. **Treno**: Il 16